# BT 5: Revision analyses

This notebook collects the analyses added in the first and second revisions of the article. The canonical implementations live in `revision_pipeline/`; sections 2 to 4 and 7 to 9 run those scripts unchanged, while sections 5 and 6 are computed directly in the notebook from the committed data files.

Prerequisite: sections 2, 3 and 6 read the outputs of `retrain.py` (reproduction order in `revision_pipeline/README.md`). The setup cell points `NB_RESULTS` at the folder written by `retrain.py --outdir`.

Contents:

1. Setup
2. Supervised cross-validation and encoder ablation (first-revision protocol)
3. SCS weight sensitivity
4. Expert agreement recomputed on the regenerated bank
5. Inter-rater reliability from the 8x30 expert vote counts
6. Threshold operating characteristics
7. Evaluation upgrades (superseded)
8. Held-out full-pipeline evaluation (first-revision protocol)
9. Manuscript tables

**Status for the second revision (October 2026).** The scripts in `revision_pipeline/` are canonical and produce the numbers reported in the article (reproduction order in `revision_pipeline/README.md`); this notebook calls them. The stored outputs of sections 1, 5 and 6 come from an earlier run of the notebook and agree with the current data; the other sections have no stored output. The display cells of sections 2 to 4 read the committed files (`revision_pipeline/cv_summary_mpnet.md`, `revision_pipeline/cv_results_*.csv`, `revision_pipeline/scs_sensitivity.md` and `results/expert/expert_report.md`), so they can be run without rerunning the scripts.

- Sections 2 and 8 run the first-revision protocols (`cv_metrics.py`, and `methodology_upgrades.py` on the seed-13 split only), which are kept for traceability. The cross-validation, encoder ablation, model selection and held-out evaluation reported in the paper (Tables 6, 7, 8 and 11) come from `supervised_eval.py` and `select_model.py` (`results/selection/`), with each family at its best configuration and seeds 13 to 17; the robustness of the selection comes from `robustness.py`.
- Section 5: the per-level Fleiss kappa ranges from -0.02 to 0.13 (Comprehension is -0.022).
- Section 6 counts entries before the top-2 collapse.

## 1. Setup

Paths are repo relative, so the notebook must be started from the repository root. `revision_pipeline/` holds the canonical scripts and committed inputs; `NB_RESULTS` is the output folder of the `retrain.py` run (it contains `winner_and_constants.json`, `structural_scan_components.csv` and the regenerated `Verb List Classified by Model.csv`).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

LEVELS = ["Kn", "Cm", "Ap", "An", "Sn", "Ev"]
N_RATERS = 8

# Repo-relative inputs
NB_PIPE = Path("revision_pipeline")
NB_EXPERT_XLSX = Path("legacy") / "Final File(with model & Expert Classification).xlsx"
assert NB_PIPE.exists() and NB_EXPERT_XLSX.exists(), "Start this notebook from the repository root."

# Outputs of retrain.py (--outdir <folder> creates <folder>/results). Adjust if needed.
NB_RESULTS = Path("results")  # BT 3 writes the retrain outputs here
if not (NB_RESULTS / "winner_and_constants.json").exists():
    NB_RESULTS = NB_PIPE / "retrain_out" / "results"

# Forward-slash strings because these are passed on %run command lines below
NB_WINNER_JSON = (NB_RESULTS / "winner_and_constants.json").as_posix()
NB_COMPONENTS = (NB_RESULTS / "structural_scan_components.csv").as_posix()
NB_BANK = NB_RESULTS / "Verb List Classified by Model.csv"
print("retrain results folder:", NB_RESULTS, "| exists:", NB_RESULTS.exists())

retrain results folder: results | exists: True


## 2. Supervised cross-validation and encoder ablation (first-revision protocol)

`revision_pipeline/cv_metrics.py` is the first-revision cross-validation protocol, kept for traceability: the models are evaluated at fixed default configurations and at the winner percentile (P50). The cross-validation reported in the paper (Table 6, each family at its best configuration) and the encoder ablation (Table 7) come from `revision_pipeline/supervised_eval.py` (`results/selection/`). The script runs repeated multi-label stratified 5-fold cross-validation (5 repeats, seeds 13 to 17) on the merged 381-verb core (607 assignments). Inside every fold it mirrors the full production decision rule: one-vs-rest fit (the code passes the frequency row weights, but with the pinned scikit-learn 1.9.0 the weighted one-vs-rest fit raises an error and the code falls back to an unweighted fit, so the weights only enter the threshold percentiles), inner out-of-fold CDF calibration, and weighted per-level thresholds at the winner percentile taken from `winner_and_constants.json`. It evaluates 14 configurations of the model zoo and two baselines, with kNN-21 (the production model) as the reference of the per-level F1 and the paired McNemar tests, then writes `cv_results_<tag>.csv` and `cv_summary_<tag>.md` into `revision_pipeline/`.

Flags: `--encoder` and `--tag` select the sentence encoder, which gives the first-revision encoder ablation. Defaults are the production encoder `sentence-transformers/all-mpnet-base-v2` with tag `mpnet`; the MiniLM and distilroberta runs are the commented lines in the next cell.

This cell fits many models and can take up to an hour on CPU. The committed `cv_results_*.csv` and `cv_summary_*.md` files in `revision_pipeline/` are the archived outputs of exactly these commands, so the display cells below work without rerunning.

In [ ]:
# Full CV under the production encoder; writes cv_results_mpnet.csv and cv_summary_mpnet.md
%run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON

# Encoder ablation, one run per encoder, same protocol:
# %run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON --encoder sentence-transformers/all-MiniLM-L6-v2 --tag minilm
# %run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON --encoder sentence-transformers/all-distilroberta-v1 --tag droberta

In [ ]:
# Main CV summary for the production encoder
display(Markdown((NB_PIPE / "cv_summary_mpnet.md").read_text(encoding="utf-8")))

In [ ]:
# Encoder ablation overview: mean over the 5 CV repeats, per model and encoder
tables = {}
for tag in ("mpnet", "minilm", "droberta"):
    f = NB_PIPE / f"cv_results_{tag}.csv"
    if f.exists():
        d = pd.read_csv(f)
        tables[tag] = d.groupby("model")[["macro_AUC", "macro_F1"]].mean().round(3)
ablation = pd.concat(tables, axis=1)
ablation.sort_values(("mpnet", "macro_F1"), ascending=False)

## 3. SCS weight sensitivity

`revision_pipeline/scs_sensitivity.py` is a short wrapper around the SCS-weight analysis of `revision_pipeline/robustness.py`. For each weight setting it re-aggregates the structural composite score SCS = wD x D - wM x M + wT x T + wC x C - wE x E (Equation 18) from the per-candidate component statistics saved by `retrain.py`, recomputes the penalized objective with the bands and penalties unchanged, re-selects each family's best configuration and applies the selection rule of `select_model.py`. The settings are each weight alone multiplied by 0.5, 0.75, 1.25 and 1.5 (20 settings) and 500 joint draws in which every weight is multiplied by a uniform factor in [0.5, 1.5]. No retraining is involved, and the cell takes about a minute. It writes a short summary to `revision_pipeline/scs_sensitivity.md`; the full per-setting table is `results/selection/robustness_scs_weights.csv`, written by `robustness.py`.

In [ ]:
%run "revision_pipeline/scs_sensitivity.py" --components $NB_COMPONENTS

In [ ]:
display(Markdown((NB_PIPE / "scs_sensitivity.md").read_text(encoding="utf-8")))

## 4. Expert agreement on the regenerated bank

`revision_pipeline/expert_recompute.py` scores the production model against the 8x30 expert votes (`legacy/Final File(with model & Expert Classification).xlsx`) and writes `results/expert/expert_report.md` with CSV tables: the Section 3.4 statistics on the 26 accepted verbs, the gate on the 29 rated pool verbs, a like-for-like comparison with the archived original-submission model on 30, 29 and 26 verbs (rejected or missing verbs counted as misses, exact paired McNemar), uniform, core-frequency and always-Application baselines, a label-permutation test that keeps the model's level distribution, per-level Fleiss kappa with the rater-subsampling curve, the score-vote Spearman correlation for both models, and the threshold sweep counted before collapse and after top-2 collapse. After normalization 'copying' resolves to the core verb 'copy' and leaves the extension, so 29 of the 30 rated verbs are candidates.

In [ ]:
%run "revision_pipeline/expert_recompute.py"

In [ ]:
# The script writes its report to results/expert/expert_report.md
display(Markdown(Path("results/expert/expert_report.md").read_text(encoding="utf-8")))

## 5. Inter-rater reliability from the expert vote counts

Eight experts rated each of the 30 sampled verbs and could endorse any subset of the six levels, or None. The committed expert file stores only per-verb vote counts (`cnt_Kn` to `cnt_Ev` and `cnt_None`, each 0 to 8), so each level is treated as its own binary rating task: every rater either endorsed the level for the verb or did not. Fleiss kappa for a binary category follows directly from those counts via observed versus expected pairwise agreement.

Kappa near zero at moderate prevalence means the experts' selections for that level are close to independent, in other words low consensus, consistent with the p_max analysis in the paper.

In [9]:
votes = pd.read_excel(NB_EXPERT_XLSX)
CATS = LEVELS + ["None"]
CNT = votes[[f"cnt_{c}" for c in CATS]].to_numpy(int)  # positive votes per verb, 0 to 8


def fleiss_kappa_binary(k, n):
    """Fleiss kappa for one binary category from per-item positive-vote counts k out of n raters."""
    k = np.asarray(k, float)
    p_obs = ((k * (k - 1) + (n - k) * (n - k - 1)) / (n * (n - 1))).mean()  # pairwise agreement
    prev = k.sum() / (k.size * n)
    p_exp = prev ** 2 + (1 - prev) ** 2
    kappa = np.nan if p_exp == 1.0 else (p_obs - p_exp) / (1 - p_exp)
    return kappa, p_obs, prev


rows = []
for j, cat in enumerate(CATS):
    kappa, p_obs, prev = fleiss_kappa_binary(CNT[:, j], N_RATERS)
    rows.append({"level": cat, "prevalence": round(prev, 2),
                 "observed agreement": round(p_obs, 2), "Fleiss kappa": round(kappa, 3)})
pd.DataFrame(rows)

,level,prevalence,observed agreement,Fleiss kappa
0,Kn,0.15,0.76,0.092
1,Cm,0.19,0.68,-0.022
2,Ap,0.29,0.60,0.020
3,An,0.18,0.73,0.081
4,Sn,0.21,0.67,0.011
5,Ev,0.19,0.73,0.132
6,None,0.28,0.61,0.039


Panel-size curve: how reliability would look with fewer raters. Raters are exchangeable, so for a panel of j raters the positive-vote count of a verb is a hypergeometric draw of j from the observed 8 votes. Each panel size uses 3000 simulated panels; the statistic is the per-level kappa averaged over the six taxonomy levels. The draw at j = 8 is deterministic and reproduces the table above. A curve that is flat by 6 to 8 raters indicates that additional raters would refine the estimates only marginally.

In [10]:
K6 = CNT[:, :6]  # the six taxonomy levels, None excluded
rng = np.random.default_rng(13)


def panel_kappas(j, iters=3000):
    vals = np.empty(iters)
    for i in range(iters):
        sub = rng.hypergeometric(K6, N_RATERS - K6, j)  # j-rater subsample per verb and level
        vals[i] = np.nanmean([fleiss_kappa_binary(sub[:, l], j)[0] for l in range(6)])
    return vals


curve = []
for j in range(2, N_RATERS + 1):
    v = panel_kappas(j)
    curve.append({"panel size": j, "mean kappa (6 levels)": round(v.mean(), 3),
                  "pct 2.5": round(np.percentile(v, 2.5), 3),
                  "pct 97.5": round(np.percentile(v, 97.5), 3)})
pd.DataFrame(curve)

,panel size,mean kappa (6 levels),pct 2.5,pct 97.5
0,2,0.041,-0.103,0.202
1,3,0.045,-0.038,0.138
2,4,0.048,-0.010,0.110
3,5,0.050,0.009,0.094
4,6,0.051,0.021,0.082
5,7,0.052,0.033,0.071
6,8,0.052,0.052,0.052


## 6. Threshold operating characteristics

User-facing view of the acceptance thresholds: the per-level hard thresholds are scaled by a factor from 1.0 down to 0.7, and a level counts as an entry when `cal_score >= scale x thr`. Entries are counted before the top-2 margin collapse, so the scale 1.0 row shows the raw threshold-pass density rather than the production bank's post-collapse 1.16 levels per accepted verb. Scale 0.85 coincides with the pipeline's soft tier. The same sweep on the 29 rated pool verbs, counted before collapse and after top-2 collapse, is in `results/expert/threshold_sweep.csv` (`revision_pipeline/expert_recompute.py`).

Columns: acceptance is the share of all candidates with at least one entry; expert precision and recall compare the predicted level set of each rated verb against its endorsed set (levels with at least 2 of 8 expert votes), averaged over the rated verbs that have at least one predicted level at that scale. These are the `*_v2_style` columns of the 'before collapse' rows of that file. The sweep in Section 3.5 of the paper averages recall over all 29 rated verbs instead, with a verb that has no predicted level counted as 0 (recall 0.38 at scale 1.0, against 0.42 here).

This table describes operating characteristics for downstream users. It must not be used to re-tune the production threshold, because the expert sample is the validation set and is far too small to tune on without contaminating the validation.

In [11]:
bank = pd.read_csv(NB_BANK)
bank["verb"] = bank["Verb"].astype(str).str.strip().str.lower()
cal = bank[[f"cal_score_{l}" for l in LEVELS]].to_numpy(float)
thr = bank[[f"thr_{l}" for l in LEVELS]].to_numpy(float)

# Rated verbs still present in the regenerated bank, with their endorsed level sets
votes["verb_clean"] = votes["Verb_clean"].astype(str).str.strip().str.lower()
rated = votes.merge(bank[["verb"]].assign(bank_row=np.arange(len(bank))),
                    left_on="verb_clean", right_on="verb", how="inner")
bank_rows = rated["bank_row"].to_numpy()
endorsed = rated[[f"cnt_{l}" for l in LEVELS]].to_numpy(int) >= 2  # at least 2 of 8 votes
print(f"{len(bank)} candidates in the bank; {len(rated)} of the 30 rated verbs matched")

1650 candidates in the bank; 29 of the 30 rated verbs matched


In [12]:
def sweep(scale):
    entries = cal >= thr * scale  # per-level entries before the top-2 collapse
    accepted = entries.any(axis=1)
    pred = entries[bank_rows]  # predicted level sets for the rated verbs
    inter = (pred & endorsed).sum(axis=1)
    has_pred = pred.sum(axis=1) > 0
    prec = (inter[has_pred] / pred.sum(axis=1)[has_pred]).mean()
    rec = (inter[has_pred] / endorsed.sum(axis=1)[has_pred]).mean()
    return {"scale": scale,
            "acceptance": round(accepted.mean(), 3),
            "levels per accepted verb": round(entries[accepted].sum() / accepted.sum(), 2),
            "expert precision": round(prec, 2),
            "expert recall": round(rec, 2)}


pd.DataFrame([sweep(s) for s in (1.00, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70)])

,scale,acceptance,levels per accepted verb,expert precision,expert recall
0,1.00,0.861,1.50,0.69,0.42
1,0.95,0.909,1.66,0.69,0.44
2,0.90,0.950,1.78,0.70,0.45
3,0.85,0.969,1.97,0.65,0.44
4,0.80,0.983,2.14,0.64,0.46
5,0.75,0.990,2.34,0.65,0.55
6,0.70,0.995,2.54,0.64,0.56


## 7. Evaluation upgrades (superseded)

The first-revision script `evaluation_upgrade.py` is kept in `legacy/round2/` (see the README there). Its gate analysis is part of `revision_pipeline/expert_recompute.py` (`results/expert/`) and its independent-encoder gap trend is computed by `revision_pipeline/bank_structure.py` (`results/bank_structure/`); the cell below runs these two scripts. Its matched-acceptance comparison used cross-validation results computed at default hyperparameters and P50 for every family and is not carried forward.

In [ ]:
%run "revision_pipeline/expert_recompute.py"
%run "revision_pipeline/bank_structure.py"

## 8. Held-out full-pipeline evaluation (first-revision protocol)

`revision_pipeline/methodology_upgrades.py` is the first-revision held-out protocol, kept for traceability. Held-out core verbs pass through the complete production pipeline (calibration, P50 thresholds, top-2 collapse) as unseen candidates. Because every held-out verb is a known Bloom verb, this measures the recall of the acceptance gate and ordinal errors that respect the order of the levels. The script evaluates five families at one default configuration each, all at P50, on one outer split (seed 13), together with a three-member ensemble that accepts a level when two of its three members accept it. The held-out results in the paper (Table 11 and the held-out criterion of Table 8) are means over five repeated splits (seeds 13 to 17) with each family at its best configuration, and come from `revision_pipeline/supervised_eval.py` (`results/selection/`).

In [ ]:
%run "revision_pipeline/methodology_upgrades.py"

## 9. Manuscript tables

`revision_pipeline/paper_results.py` runs no model. It reads the committed outputs of the reproduction sequence, checks them against each other and writes every table and number of the manuscript to `results/paper_tables.md` and `results/paper_numbers.json`. The robustness of the selection to other ways of combining the three criteria, to the SCS weights and to the held-out split is computed by `revision_pipeline/robustness.py` (`results/selection/robustness_report.md`).

In [ ]:
%run "revision_pipeline/paper_results.py"